<a href="https://colab.research.google.com/github/ezh2020/FloodRoad-SAM3/blob/main/FloodRoad_SAM3_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FloodRoad-SAM3 Real Colab Runner

This notebook runs the real SpaceNet 8 flooded-road experiment only. It does not create or evaluate toy data, and it pulls code directly from the public GitHub repository.

Requirements before running: a GPU runtime, Hugging Face access to `facebook/sam3`, a Colab Secret named `HF_TOKEN`, and either existing SpaceNet 8 data under `/content/spacenet8/raw` or enough time/storage to download the public bucket.


## 1. Clone public repo and install dependencies


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ezh2020/FloodRoad-SAM3.git"
REPO_DIR = Path("/content/FloodRoad-SAM3")

def run(cmd, **kwargs):
    print("+", " ".join(map(str, cmd)), flush=True)
    result = subprocess.run(cmd, text=True, **kwargs)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(map(str, cmd))}")
    return result

%cd /content
shutil.rmtree(REPO_DIR, ignore_errors=True)
run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
%cd /content/FloodRoad-SAM3
!git rev-parse --short HEAD
!python -m pip install -q -r requirements.txt
!python -m py_compile run_colab_real.py train.py evaluate.py data/preprocess_sn8.py data/preprocess.py data/dataset.py models/*.py metrics.py utils.py


## 2. Authenticate Hugging Face for official SAM3 weights


In [ ]:
import os
from google.colab import userdata

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN") or userdata.get("HF_TOKEN")
assert token, "Missing Colab Secret named HF_TOKEN. Add it after your Hugging Face account has access to facebook/sam3."
os.environ["HF_TOKEN"] = token
os.environ["HUGGING_FACE_HUB_TOKEN"] = token
print("HF token loaded:", bool(token))


## 3. Validate runtime, public data, and SAM3 API


In [ ]:
import importlib
import os
import subprocess
import sys
from pathlib import Path

print("===== GPU =====")
subprocess.run(["nvidia-smi", "-L"], check=False)
subprocess.run(["nvidia-smi"], check=False)

print("\n===== Python / Torch =====")
print(sys.version)
import torch
print("torch", torch.__version__, "cuda available", torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA GPU is not available. Switch Colab Runtime > Change runtime type > GPU."
print("device", torch.cuda.get_device_name(0))

print("\n===== Key package imports =====")
for name in ["numpy", "rasterio", "cv2", "networkx", "geopandas", "shapely", "yaml", "thop"]:
    try:
        mod = importlib.import_module(name)
        print(name, getattr(mod, "__version__", "ok"))
    except Exception as exc:
        raise RuntimeError(f"Failed to import {name}: {exc}")

print("\n===== SpaceNet 8 public tarballs =====")
tarball_cmd = [
    "aws", "s3", "ls",
    "s3://spacenet-dataset/spacenet/SN8_floods/tarballs/",
    "--no-sign-request",
]
result = subprocess.run(tarball_cmd, text=True, capture_output=True)
if result.returncode != 0:
    print(result.stdout)
    print(result.stderr)
    raise RuntimeError("Could not list SpaceNet 8 public tarballs. Check Colab network access.")
tarball_lines = [line for line in result.stdout.splitlines() if line.strip()]
for line in tarball_lines:
    if "Louisiana" in line or "louisiana" in line:
        print(line)
default_tarball = "Louisiana-East_Training_Public.tar.gz"
available = [line.split()[-1] for line in tarball_lines]
available_tarballs = [name for name in available if name.endswith(".tar.gz")]
assert available_tarballs, "No .tar.gz files were listed under the SpaceNet 8 tarballs prefix."
if default_tarball in available:
    sn8_tarball = default_tarball
else:
    candidates = [name for name in available_tarballs if "Louisiana" in name and "Training" in name]
    east = [name for name in candidates if "East" in name]
    sn8_tarball = (east or candidates or available_tarballs)[0]
os.environ["SN8_TARBALL"] = sn8_tarball
print("Selected SN8 tarball:", sn8_tarball)

print("\n===== SAM3 install/import probe =====")
try:
    import sam3  # noqa: F401
    print("sam3 already importable")
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/sam3.git"], check=True)

sam3_candidates = [
    ("sam3", "build_sam3_image_model"),
    ("sam3.model_builder", "build_sam3_image_model"),
    ("sam3", "build_sam3"),
    ("segment_anything_3", "build_sam3"),
    ("segment_anything_3", "sam_model_registry"),
]
sam3_hits = []
for module_name, attr_name in sam3_candidates:
    try:
        mod = importlib.import_module(module_name)
        attr = getattr(mod, attr_name)
        print("OK", module_name, attr_name, attr)
        sam3_hits.append((module_name, attr_name))
    except Exception as exc:
        print("NO", module_name, attr_name, repr(exc))
assert sam3_hits, "SAM3 installed, but none of the adapter's known entry points were found. Paste this cell output back for adapter update."

print("\n===== Repository sanity =====")
subprocess.run(["git", "remote", "-v"], check=False)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)
print("Validation passed. The next cell starts the formal real-data run.")


## 4. Run real SpaceNet 8 experiment


In [ ]:
!python run_colab_real.py \
  --config configs/default.yaml \
  --raw-root /content/spacenet8/raw \
  --processed-root /content/spacenet8/processed \
  --output-dir /content/floodroad_runs/default \
  --sn8-tarball "$SN8_TARBALL"


## 5. Show result tables and run metadata


In [ ]:
from pathlib import Path

out = Path("/content/floodroad_runs/default")
processed = Path("/content/spacenet8/processed")
for path in [
    out / "real_run.yaml",
    processed / "preprocess_summary.json",
    out / "rl_samples.json",
    out / "accuracy_table.md",
    out / "efficiency_table.md",
]:
    print(f"\n===== {path} =====")
    print(path.read_text()[:8000] if path.exists() else "missing")

print("\n===== output files =====")
for path in sorted(out.rglob("*")):
    if path.is_file():
        print(path)

# Backward-compatible short table echo.
for name in ["accuracy_table.md", "efficiency_table.md"]:
    path = Path("/content/floodroad_runs/default") / name
    print(f"\n===== {name} =====")
    print(path.read_text() if path.exists() else "missing")

